[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/rl_cartpole_sb3.ipynb)

# CartPole with Stable-Baselines3 (PPO)

The same CartPole problem of the `rl_cartpole` notebook, solved with the PPO (Proximal Policy Optimization) algorithm of Stable-Baselines3. The maximum reward is 500 (the pole stays upright for 500 steps).

[Stable-Baselines3](https://stable-baselines3.readthedocs.io/) provides reliable implementations of reinforcement learning algorithms. The steps are always the same: create the environment, create the model, train it, and evaluate it.

## Install packages

In [ ]:
!pip install -q stable-baselines3 "gymnasium[classic-control]" moviepy

## Import libraries

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from gymnasium.utils.save_video import save_video
from IPython.display import Video, display

## 1. Create the environment

In [ ]:
env = gym.make("CartPole-v1")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

## 2. Create the model and 3. Train the agent

The training statistics are printed periodically (e.g., `ep_rew_mean` is the average reward of the last episodes) and logged for TensorBoard in the `tb/cartpole` folder.

In [ ]:
model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="tb/cartpole")

model.learn(total_timesteps=10_000)

Save the trained agent, and load it again (just to illustrate how to do it).

In [ ]:
model.save("cartpole_sb3_agent")

In [ ]:
del model

In [ ]:
model = PPO.load("cartpole_sb3_agent", env=env)

## 4. Evaluate the agent

Average reward over 10 episodes, choosing the actions deterministically.

In [ ]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10, deterministic=True)
print(f"Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

## Run a demo

Record one episode as a video and show it. (Locally, you can watch it live in a window, creating the environment with `render_mode="human"`.)

In [ ]:
def record_episode(model, name, seed=0):
    env = gym.make("CartPole-v1", render_mode="rgb_array_list")
    obs, _ = env.reset(seed=seed)
    done = False
    total_reward = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        done = terminated or truncated
    save_video(env.render(), "videos", fps=env.metadata["render_fps"], name_prefix=name,
               episode_trigger=lambda episode: True)
    env.close()
    print(f"Total reward: {total_reward:.2f}")
    return Video(f"videos/{name}-episode-0.mp4", embed=True)


display(record_episode(model, "cartpole_sb3_agent"))

## TensorBoard

Uncomment to see the training curves in TensorBoard.

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir tb/cartpole